In [1]:
import os
import pandas as pd

# Use GPU 2
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["WANDB_MODE"] = "disabled"

import torch

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

os.chdir('/shared/4/projects/research-jam-2024/')

# Settings
MODEL_PATH = 'models/fine-tuned/roberta-base/checkpoint-181'
DATA_FP = 'working-dir/monthly-posts-cleaned/en.2009-12.tsv'
OUT_DIR = 'working-dir/for-pred/preds/'

2024-06-11 20:35:38.723645: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


**Load in the data**

In [2]:
df = pd.read_csv('working-dir/for-pred/preds/all_data_for_pred.tsv', sep='\t')
df = df[['message_id','message_body_clean']].\
    rename(columns = {'message_id':'id', 'message_body_clean':'text'})
df.head()

,id,text
0,<a0M6haL4diNK63cR@example.com>,<gmane_tag_salutation> <gmane_tag_quotation_ma...
1,<V9TRyfzalFkUJxPr@example.com>,<gmane_tag_quotation_marker> <gmane_tag_quotat...
2,<LLUOc/zPQxmARvI9@example.com>,<gmane_tag_quotation_marker> Interesting. I ca...
3,<c1Eq7hEbuNEJrBoR@example.com>,Many countries have disclosure laws concerning...
4,<hXMzR7ywGdPi6A4D@example.com>,<gmane_tag_quotation_marker> <gmane_tag_quotat...


**Load in the model**

In [3]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model = model.to('cuda')

**Code to generate predictions**

In [ ]:
labels = ['Sharing', 'Requesting', 'Promising', 'Personal', 'Pleasantries', 'Spam', 'ResponseExpected']


def run_predictions(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=350, return_tensors="pt")
    inputs = {k: v.to('cuda') for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    predictions = torch.sigmoid(outputs.logits)
    predictions = predictions.cpu().numpy()

    predictions_dicts = []
    for pred in predictions:
        pred_dict = {label: float(pred_val) for label, pred_val in zip(labels, pred)}
        predictions_dicts.append(pred_dict)

    return predictions_dicts

model.eval()
batch_size = 256
predictions_list = []
for i in tqdm(range(0, len(df), batch_size), desc="Processing text batches"):
    batch_texts = df['text'][i:i+batch_size].tolist()
    batch_predictions = run_predictions(batch_texts)
    predictions_list.extend(batch_predictions)

Processing text batches:   7%|██████                                                                                 | 160/2280 [03:36<46:37,  1.32s/it]

**Run on the dataframe**

In [ ]:
predictions_df = pd.DataFrame(predictions_list)
predictions_df.head()

**Save the results**

In [ ]:
final_df = pd.concat([df['id'], predictions_df], axis=1)
final_df.head()

In [ ]:
# Specify your output file path
output_csv_fp = "working-dir/" + "roberta-preds.csv"
final_df.to_csv(output_csv_fp, index=False)